# Week 1 综合复习 + 阶段自测

A 组：Python 基础

1. 给定 nums = [4, 2, 7, 1, 9, 3, 8, 5],不用 sorted()/sort(),返回第二大的数。(提示:可以遍历一次,维护"最大"和"第二大"两个变量)

In [82]:
nums = [4, 2, 7, 1, 9, 3, 8, 5]
mx, second_mx = nums[0], nums[0]
for num in nums:
    if num >= mx:
        second_mx = mx
        mx = num
    elif second_mx <= num < mx:
        second_mx = num
        
print(second_mx)

8


参考答案

第 1 题:逻辑有隐藏 bug,这次"碰巧"对了

假设输入的例子为[3, 1, 2], 就会输出错误

In [101]:
nums = [3, 1, 2]
mx, second_mx = nums[0], nums[0]
for num in nums:
    if num >= mx:
        second_mx = mx
        mx = num
    elif second_mx <= num < mx:
        second_mx = num
        
print(second_mx)

3


bug 的根源在哪

根源是初始化:second_mx = nums[0]。

second_mx 一开始就被设成了 3,而 list 里根本没有比 3 更小、又能更新它的机会——因为你的 elif 条件是 second_mx <= num,要求新数必须 大于等于 当前的 second_mx(3) 才能更新它。可 1 和 2 都比 3 小,所以永远进不去,second_mx 卡在 3 不动。

换句话说:你把 second_mx 初始化成了一个"太大"的值,导致比它小的合理候选数永远无法替换它。

正确的初始化思路

第二大的数,初始时应该是"还不知道",所以要初始化成一个绝对不可能误导后续判断的值 —— 负无穷:

In [103]:
nums = [4, 2, 7, 1, 9, 3, 8, 5]

mx = float('-inf')          # 负无穷,任何数都比它大
second_mx = float('-inf')

for num in nums:
    if num > mx:
        second_mx = mx      # 原来的最大值"退位"成第二
        mx = num            # 新的最大值
    elif num > second_mx and num != mx:
        second_mx = num

print(second_mx)

8


2. 给定一个 list ['a', 'b', 'a', 'c', 'b', 'a', 'd'],返回出现次数恰好为 1 次的所有元素 → ['c', 'd']。

In [83]:
chars = ['a', 'b', 'a', 'c', 'b', 'a', 'd']
counters = {}
lst = []

for char in chars:
    counters[char] = counters.get(char, 0) + 1

for char, freq in counters.items():
    if freq == 1:
        lst.append(char)

print(lst)

['c', 'd']


3. 给定两个 dict:

q1 = {'Alice': 100, 'Bob': 80, 'Charlie': 120}
q2 = {'Alice': 150, 'Bob': 90, 'David': 200}

合并成一个 dict,相同 key 的 value 相加,不同 key 保留:

期望:{'Alice': 250, 'Bob': 170, 'Charlie': 120, 'David': 200}

In [84]:
q1 = {'Alice': 100, 'Bob': 80, 'Charlie': 120}
q2 = {'Alice': 150, 'Bob': 90, 'David': 200}

q_combined = {}
for name, nums in q1.items():
    q_combined[name] = q_combined.get(name, 0) + nums

for name, nums in q2.items():
    q_combined[name] = q_combined.get(name, 0) + nums

print(q_combined)


{'Alice': 250, 'Bob': 170, 'Charlie': 120, 'David': 200}


参考答案

In [96]:
from collections import Counter
q_combined = Counter(q1) + Counter(q2)   # Counter 支持直接相加!

print(q_combined)

Counter({'Alice': 250, 'David': 200, 'Bob': 170, 'Charlie': 120})


Counter 是 dict 的"计数特化版",相加时自动合并同 key。以后会越来越常用。

4.  给定订单数据,用一行列表推导式,提取所有金额 > 100 的订单的 order_id,并按 order_id 排序:

orders = [
    
    {'order_id': 'B003', 'amount': 250},

    {'order_id': 'B001', 'amount': 80},

    {'order_id': 'B002', 'amount': 150},

    {'order_id': 'B004', 'amount': 300},
]

期望:['B002', 'B003', 'B004']

In [85]:
orders = [
    {'order_id': 'B003', 'amount': 250},
    {'order_id': 'B001', 'amount': 80},
    {'order_id': 'B002', 'amount': 150},
    {'order_id': 'B004', 'amount': 300},
]

print(sorted([order['order_id'] for order in orders if order['amount'] > 100]))

['B002', 'B003', 'B004']


B 组：SQL 聚合

In [86]:
import duckdb

%load_ext sql
%sql duckdb://

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


5. 算出每个国家的平均客单价(ROUND 保留 2 位),按平均客单价降序。

In [87]:
duckdb.sql("""
    SELECT
        country,
        ROUND(AVG(total), 2) as revenue_per_order
    FROM '../data/sales.csv'
    GROUP BY country
    ORDER BY revenue_per_order DESC
""")

┌─────────┬───────────────────┐
│ country │ revenue_per_order │
│ varchar │      double       │
├─────────┼───────────────────┤
│ UK      │           2714.81 │
│ France  │           2602.06 │
│ China   │           2503.16 │
│ US      │            2404.9 │
│ Germany │           2182.12 │
└─────────┴───────────────────┘

6. 找出总销售额排名前 3 的产品,显示产品名和总销售额。

In [88]:
duckdb.sql("""
    SELECT
        product,
        SUM(total) AS revenue
    FROM '../data/sales.csv'
    GROUP BY product
    ORDER BY revenue DESC
    LIMIT 3
""")

┌──────────┬─────────┐
│ product  │ revenue │
│ varchar  │ int128  │
├──────────┼─────────┤
│ Phone    │  257831 │
│ Keyboard │  225111 │
│ Mouse    │  216155 │
└──────────┴─────────┘

7. 算出每个 customer_id 的订单数,只保留下单次数在 50 到 70 之间(含)的客户,按订单数降序。 

💡 提醒:HAVING 里写聚合函数本身,别用别名(Day 5 的教训)。

In [89]:
duckdb.sql("""
    SELECT
        customer_id,
        COUNT(*) AS order_count
    FROM '../data/sales.csv'
    GROUP BY customer_id
    HAVING 50 <= COUNT(*) AND COUNT(*) <= 70
    ORDER BY order_count DESC
""")

┌─────────────┬─────────────┐
│ customer_id │ order_count │
│   varchar   │    int64    │
├─────────────┼─────────────┤
│ C003        │          64 │
│ C006        │          62 │
│ C007        │          58 │
│ C005        │          51 │
└─────────────┴─────────────┘

8. 按 category 分组,算出每个品类的总销售额,以及该品类销售额占全部销售额的百分比(保留 1 位小数)。

💡 提示:全部销售额是一个固定数字,你可以先单独查出来(比如 1267716),然后写进 SQL 里当除数。进阶做法明天讲。

In [90]:
duckdb.sql("""
    SELECT
        category,
        SUM(total) AS revenue,
        ROUND(SUM(total) * 100 / 1267716, 1) AS revenue_percent   
    FROM '../data/sales.csv'
    GROUP BY category
""")

┌───────────┬─────────┬─────────────────┐
│ category  │ revenue │ revenue_percent │
│  varchar  │ int128  │     double      │
├───────────┼─────────┼─────────────────┤
│ Accessory │  441266 │            34.8 │
│ Computer  │  363933 │            28.7 │
│ Audio     │  204686 │            16.1 │
│ Mobile    │  257831 │            20.3 │
└───────────┴─────────┴─────────────────┘

9. 找出在 UK 这个国家,哪个品类的订单数最多。只返回 1 行(品类名 + 订单数)。

In [91]:
duckdb.sql("""
    SELECT
        category,
        COUNT(*) AS order_count
    FROM '../data/sales.csv'
    WHERE country = 'UK'
    GROUP BY category
    ORDER BY order_count DESC
    LIMIT 1
""")

┌───────────┬─────────────┐
│ category  │ order_count │
│  varchar  │    int64    │
├───────────┼─────────────┤
│ Accessory │          71 │
└───────────┴─────────────┘

C 组：综合应用

10. (SQL)月度趋势分析:order_date 是 'YYYY-MM-DD' 格式。按"年-月"分组,算出每个月的订单数和总销售额。

💡 提示:SUBSTRING(order_date, 1, 7)可以取出 '2024-01' 这样的"年-月"。或者 DuckDB 支持 strftime。先用 SUBSTRING 即可。

In [92]:
duckdb.sql("""
    SELECT
        STRFTIME(order_date, '%Y-%m') AS year_month,
        COUNT(*) AS order_count,
        SUM(total) AS revenue
    FROM '../data/sales.csv'
    GROUP BY year_month
    ORDER BY year_month
""")

┌────────────┬─────────────┬─────────┐
│ year_month │ order_count │ revenue │
│  varchar   │    int64    │ int128  │
├────────────┼─────────────┼─────────┤
│ 2024-01    │          43 │   99365 │
│ 2024-02    │          40 │   81795 │
│ 2024-03    │          42 │  133371 │
│ 2024-04    │          41 │  123882 │
│ 2024-05    │          42 │   71176 │
│ 2024-06    │          41 │  119061 │
│ 2024-07    │          43 │  101271 │
│ 2024-08    │          42 │  104388 │
│ 2024-09    │          41 │  103183 │
│ 2024-10    │          42 │  117265 │
│ 2024-11    │          41 │  105984 │
│ 2024-12    │          42 │  106975 │
└────────────┴─────────────┴─────────┘
  12 rows                  3 columns

11. (Python)给定一段销售文本日志,统计每个产品被提到的次数:

log = "Laptop sold. Phone sold. Laptop returned. Mouse sold. Phone sold. Laptop sold."

每个句子以 '. ' 分隔,每句的第一个词是产品名

期望:{'Laptop': 3, 'Phone': 2, 'Mouse': 1}

In [93]:
log = "Laptop sold. Phone sold. Laptop returned. Mouse sold. Phone sold. Laptop sold."
log_clean = log.replace('.',' ').split()
counter = {}

for i in range(0,len(log_clean), 2):
    counter[log_clean[i]] = counter.get(log_clean[i], 0) + 1

print(counter) 

{'Laptop': 3, 'Phone': 2, 'Mouse': 1}


参考答案

你用 log.replace('.', ' ').split() 把句号换成空格再切分,然后 range(0, len, 2) 每隔一个取产品名 —— 利用了"产品名 + 动词"两两成对的规律。聪明。

⭐ 不过这个解法依赖"每句正好两个词"的假设。如果某句是 "Gaming Laptop sold"(产品名两个词),就会错位。更稳健的写法是按句子切分,再取每句第一个词:

In [100]:
log = "Laptop sold. Phone sold. Laptop returned. Mouse sold. Phone sold. Laptop sold."
counter = {}
for sentence in log.split('.'):        # 按句号切分成句子
    sentence = sentence.strip()         # 去掉两边空格
    if sentence:                        # 跳过空字符串(最后一个句号后面是空的)
        product = sentence.split()[0]   # 每句第一个词
        counter[product] = counter.get(product, 0) + 1
print(counter)

{'Laptop': 3, 'Phone': 2, 'Mouse': 1}


12. (SQL,挑战题)找出"消费高于平均水平"的客户:先算出所有客户的平均消费额是多少,再找出总消费额高于这个平均值的客户。

💡 这道题有难度。提示:你可以分两步——先单独查出"平均每个客户消费多少",把那个数字记下来,再写第二个查询用 HAVING SUM(total) > 那个数字。明天会教你怎么用子查询一步到位,今天先用"两步走"。

In [94]:
# 我没有思路

参考答案

"两步走"做法(今天的要求):

先单独跑一个查询,算出"平均每个客户消费多少":

In [97]:
duckdb.sql("""
    SELECT AVG(customer_total) AS avg_per_customer 
    FROM (
        SELECT customer_id, SUM(total) AS customer_total
        FROM '../data/sales.csv' GROUP BY customer_id)
""")

┌──────────────────┐
│ avg_per_customer │
│      double      │
├──────────────────┤
│         158464.5 │
└──────────────────┘

然后第二个查询:

In [98]:
duckdb.sql("""
    SELECT customer_id, 
           SUM(total) AS total_spent
    FROM '../data/sales.csv'
    GROUP BY customer_id
    HAVING SUM(total) > 158464.5
    ORDER BY total_spent DESC
""")

┌─────────────┬─────────────┐
│ customer_id │ total_spent │
│   varchar   │   int128    │
├─────────────┼─────────────┤
│ C006        │      198806 │
│ C001        │      184001 │
│ C008        │      164484 │
│ C003        │      160212 │
└─────────────┴─────────────┘

而"一步到位"的做法(明天教的子查询)长这样:

In [99]:
duckdb.sql("""
    SELECT customer_id, 
           SUM(total) AS total_spent
    FROM '../data/sales.csv'
    GROUP BY customer_id
    HAVING SUM(total) > (
        SELECT AVG(customer_total)
        FROM (
             SELECT customer_id, SUM(total) AS customer_total
             FROM '../data/sales.csv'
             GROUP BY customer_id
        )
    )
    ORDER BY total_spent DESC
""")

┌─────────────┬─────────────┐
│ customer_id │ total_spent │
│   varchar   │   int128    │
├─────────────┼─────────────┤
│ C006        │      198806 │
│ C001        │      184001 │
│ C008        │      164484 │
│ C003        │      160212 │
└─────────────┴─────────────┘

Week 1 自测清单

Python 部分

我能说出 list 和 tuple 的 3 个区别

我只知道list可改而tuple不可以

1. 可变性:list 可变,tuple 不可变 ✓(你答对了)
2. 能否作为 dict 的 key:tuple 可以,list 不可以(因为 dict key 必须不可变)
3. 语义/性能:tuple 表示"一组固定的东西"(如坐标、数据库一行记录),且比 list 略快、略省内存

我知道为什么 b = a(a 是 list)之后改 b 会影响 a

因为两个变量存在同一个内存地址

我能不假思索写出"计数"的字典模式

可以做到

我能解释列表推导式里 if 在 for 前和 for 后的区别

前者是先条件筛选再迭代 后者是先迭代再筛选

[x for x in seq if cond] —— if 在后,是过滤,decide 元素保不保留

[x if cond else y for x in seq] —— if 在前,是三元表达式,decide 每个元素输出成什么(所有元素都保留)

我能写出多关键字排序 sorted(key=lambda x: (a, -b))

可以

SQL 部分

我能背出 SQL 的执行顺序(7 个子句)

可以 SELECT → FROM → WHERE → GROUP BY → HAVING → ORDER BY → LIMIT

我能解释 WHERE 和 HAVING 的区别

WHERE是在表内的原数据筛选 HAVING是对处理过的数据进行筛选

我知道 COUNT(*)、COUNT(col)、COUNT(DISTINCT col) 的区别

第一个是查询总行数 第二个统计col中的非空行数 第三个统计col的去重行数

我知道 HAVING 里为什么不该用别名

因为在处理HAVING的时候 别名的变量还没有生成

我能说出 GROUP BY 的"黄金法则"

SELECT里出现的列，要么在GROUP BY里，要么被聚合函数包着